# Money and Decimal

Money in Paxman is **Decimal** + ISO-4217 currency + precision. The `MONEY` field type is first-class — it carries an `amount` (as `Decimal`), a `currency_code`, and a `precision`.

> **Important:** Floats are silently mapped to `DECIMAL` by the Pydantic adapter (see issue #61, fixed in v1.0.2). Always use `Decimal` for money amounts.

In [ ]:
from decimal import Decimal

from pydantic import BaseModel, Field

import paxman
import paxman.contract.adapters.pydantic


class Invoice(BaseModel):
    supplier: str
    total: Decimal = Field(..., description="Total invoice amount")
    currency: str = Field(default="MYR", description="ISO-4217 currency code")


sample = """
ACME Hardware Supplies
Total: RM 500.50
"""
result = paxman.normalize(sample, Invoice)
print(f"Status: {result.status.name}")
print(f"Normalized: {result.normalized_data}")

In [ ]:
from pydantic import BaseModel as BM


class FloatModel(BM):
    amount: float


class DecimalModel(BM):
    amount: Decimal


float_result = paxman.normalize("amount: 500.50", FloatModel)
decimal_result = paxman.normalize("amount: 500.50", DecimalModel)
print(
    f"Float field type:    {type(float_result.normalized_data['amount']).__name__} = {float_result.normalized_data['amount']}"
)
print(
    f"Decimal field type:  {type(decimal_result.normalized_data['amount']).__name__} = {decimal_result.normalized_data['amount']}"
)

## Key rules

1. **Never use `float` for money** — use `Decimal` from the `decimal` module.
2. **Currency is ISO-4217** — 3-letter uppercase codes (MYR, USD, EUR, etc.).
3. **Precision** is tracked explicitly in Paxman's internal `MoneyValue` structure.

> **Reference:** The Paxman contract subsystem has a first-class `MoneyValue` type (`paxman.contract.canonical.MoneyValue`) that carries `amount (Decimal) + currency + precision`. Money arithmetic is handled by the `reconciler/money.py` module, which is the only module allowed to `import decimal`.

## Try it yourself

- Create a model with multiple money fields and observe the precision behavior.
- What happens if you pass a negative `Decimal` value?
- Try a cross-currency invoice with two money fields in different currencies.